# Phase 4 Stage 4 — Ensemble (v3 + v4-resume + v4-scratch + v8)

**Muc dich:** Ridge regression trong log-space de blend cac mo hinh doc lap:
- `v3` — QLoRA Qwen3.5-4B (85K data, RMSLE ~0.44)
- `v4_resume` — v3 resumed, NEFTune + LR mem (RMSLE ~0.40-0.42)
- `v4_scratch` — QLoRA Qwen3.5-4B (271K augmented, DoRA+RSLoRA, RMSLE ~0.36-0.40)
- `v8` — Day4 PhoBERT-large stacked (RMSLE 0.4004)

**Pipeline:**
1. Load val predictions tu JSON (moi mo hinh da save pred + true)
2. Alignment check (N=3,926, cung order)
3. Fit Ridge regression: `y = sum(w_i * log(pred_i)) + b` tren val
4. Eval ensemble RMSLE vs tung model
5. Run tung mo hinh tren test (3,872) → ensemble → save

**Khong can GPU** neu tat ca prediction files da co san.
GPU can: khi phai chay Qwen inference (v3/v4-resume/v4-scratch co adapter nhu khong co val preds file).

In [ ]:
import os
import re
import sys
import gc
import json
import time
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

NOTEBOOK_DIR = Path("__file__").parent if "__file__" in dir() else Path(".")
sys.path.insert(0, str(NOTEBOOK_DIR))
from utils.evaluator import compute_metrics, plot_predictions

import numpy as np
import sklearn
print(f"numpy  : {np.__version__}")
print(f"sklearn: {sklearn.__version__}")
print("Imports OK")


In [ ]:
# --- Paths to pre-computed val prediction files ---
# Moi file: [{"idx": i, "pred_vnd": int, "true_vnd": int}, ...] cho 3926 val items
RESULTS_DIR = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

PRED_FILES = {
    "v3":         RESULTS_DIR / "v3_val_predictions.json",
    "v4_resume":  RESULTS_DIR / "v4_resume_val_predictions.json",
    "v4_scratch": RESULTS_DIR / "v4_scratch_val_predictions.json",
    "v8":         RESULTS_DIR / "v8_val_predictions.json",
}

# Test prediction files (populated in Section 5)
TEST_PRED_FILES = {
    "v3":         RESULTS_DIR / "v3_test_predictions.json",
    "v4_resume":  RESULTS_DIR / "v4_resume_test_predictions.json",
    "v4_scratch": RESULTS_DIR / "v4_scratch_test_predictions.json",
    "v8":         RESULTS_DIR / "v8_test_predictions.json",
}

ENSEMBLE_RESULTS_FILE = RESULTS_DIR / "ensemble_results.json"

# HF dataset cho val/test prompts + ground truth
DATASET_NAME = "SeanSunny/items_prompts_tv_3"  # val/test giu nguyen tu tv_3

# Qwen model configs (dung khi phai chay inference)
QWEN_MODELS = {
    "v3":         "SeanSunny/qwen3.5-4b-vn-pricer-v3",
    "v4_resume":  "SeanSunny/qwen3.5-4b-vn-pricer-v4-resume",
    "v4_scratch": "SeanSunny/qwen3.5-4b-vn-pricer-v4-scratch",
}

SEED = 42
MAX_NEW_TOKENS = 4
PRED_CLAMP_MIN = 5
PRED_CLAMP_MAX = 1000
PARSE_REGEX    = r"[-+]?\d*\.\d+|\d+"

random.seed(SEED)
np.random.seed(SEED)

for name, p in PRED_FILES.items():
    status = "EXISTS" if p.exists() else "MISSING"
    print(f"  {name:<12}: {status} — {p}")


## 1. Load dataset (val + test ground truth + prompts)

Val/test giu nguyen tu `items_prompts_tv_3` — KHONG dung tv_4 de tranh data leak.

In [ ]:
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login

env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK")
else:
    login()

ds = load_dataset(DATASET_NAME)
val_ds  = ds["val"]
test_ds = ds["test"]
print(f"Val : {len(val_ds):,} | Test: {len(test_ds):,}")

val_trues_vnd  = np.array([item["price_vnd_true"] for item in val_ds],  dtype=float)
test_trues_vnd = np.array([item["price_vnd_true"] for item in test_ds], dtype=float)
print(f"Val  RMSLE baseline (predict mean): ")
mean_pred = np.full_like(val_trues_vnd, np.mean(val_trues_vnd))
from utils.evaluator import rmsle
print(f"  {rmsle(val_trues_vnd, mean_pred):.4f} (predict train mean)")


## 2. Load val predictions

Neu file ton tai, load truc tiep. Neu MISSING, chay cell inference tuong ung.

In [ ]:
def load_preds_from_file(path: Path) -> np.ndarray | None:
    """Load prediction array [pred_vnd, ...] from JSON file. Return None if missing."""
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    preds = np.array([d["pred_vnd"] for d in data], dtype=float)
    return preds

def load_trues_from_file(path: Path) -> np.ndarray | None:
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return np.array([d["true_vnd"] for d in data], dtype=float)

val_preds = {}
for name, path in PRED_FILES.items():
    arr = load_preds_from_file(path)
    if arr is not None:
        val_preds[name] = arr
        m = rmsle(val_trues_vnd[:len(arr)], arr)
        print(f"Loaded {name:<12}: {len(arr):,} preds, RMSLE={m:.4f}")
    else:
        print(f"MISSING {name:<12}: {path}")

print(f"\nModels ready for ensemble: {list(val_preds.keys())}")


## 2b. [OPTIONAL] Qwen inference — chay neu prediction file MISSING

**Chi chay khi file predictions chua ton tai.** Yeu cau GPU.

Cell nay chay inference cho tat ca Qwen models co adapter HF nhung chua co val preds file.

In [ ]:
# --- [SKIP neu da co tat ca files] ---
missing_qwen = [k for k in QWEN_MODELS if k not in val_preds]
if not missing_qwen:
    print("All Qwen predictions available. Skip this cell.")
else:
    print(f"Computing inference for: {missing_qwen}")
    import torch
    import bitsandbytes as bnb
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    assert torch.cuda.is_available(), "GPU required for inference."

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )

    def run_qwen_inference(hf_repo: str, split_ds, save_path: Path) -> np.ndarray:
        """Load adapter from HF Hub, run inference on split_ds, save + return preds."""
        tok = AutoTokenizer.from_pretrained(hf_repo, trust_remote_code=True)
        tok.pad_token = tok.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3.5-4B-Base",
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            dtype=torch.bfloat16,
        )
        for _m in base.modules():
            if isinstance(_m, torch.nn.Conv1d):
                _m.to(torch.bfloat16)
        model = PeftModel.from_pretrained(base, hf_repo, is_trainable=False)
        model.eval()

        preds_vnd, dump = [], []
        for i, item in enumerate(tqdm(split_ds, desc=hf_repo[-20:])):
            inputs = tok(item["prompt"], return_tensors="pt").to("cuda")
            with torch.inference_mode():
                out = model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False, pad_token_id=tok.eos_token_id,
                )
            gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            m = re.search(PARSE_REGEX, gen)
            pk = max(PRED_CLAMP_MIN, min(int(float(m.group())), PRED_CLAMP_MAX)) if m else 0
            preds_vnd.append(pk * 1000)
            dump.append({"idx": i, "pred_vnd": pk * 1000, "true_vnd": int(item["price_vnd_true"])})

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(dump, f, ensure_ascii=False)
        print(f"Saved {len(dump)} preds -> {save_path}")

        del model, base
        gc.collect()
        torch.cuda.empty_cache()
        return np.array(preds_vnd, dtype=float)

    for name in missing_qwen:
        preds = run_qwen_inference(QWEN_MODELS[name], val_ds, PRED_FILES[name])
        val_preds[name] = preds
        m = rmsle(val_trues_vnd, preds)
        print(f"Computed {name}: RMSLE={m:.4f}")

print(f"\nModels ready: {list(val_preds.keys())}")


## 2c. [OPTIONAL] v8 predictions — chay neu v8_val_predictions.json MISSING

**Option A (de nhat):** Mo `day4/day4_dl_models_v8.ipynb`, chay toan bo, save predictions:
```python
# Them vao cuoi day4 notebook:
import json
dump_val  = [{"idx": i, "pred_vnd": int(v), "true_vnd": int(t)}
             for i, (v, t) in enumerate(zip(ALL_PREDS_VAL["v8: Weighted Blend"] * 1000,
                                             val_prices_np * 1000))]
dump_test = [{"idx": i, "pred_vnd": int(v), "true_vnd": int(t)}
             for i, (v, t) in enumerate(zip(ALL_PREDS_TEST["v8: Weighted Blend"] * 1000,
                                             test_prices_np * 1000))]
with open("path/to/fine_tune_qwen/results/v8_val_predictions.json", "w") as f:
    json.dump(dump_val, f)
with open("path/to/fine_tune_qwen/results/v8_test_predictions.json", "w") as f:
    json.dump(dump_test, f)
```

**Option B:** Bo qua v8, chay ensemble 3 Qwen models.

In [ ]:
if "v8" not in val_preds:
    print("WARNING: v8 predictions not found.")
    print("Ensemble se chay voi cac model: ", list(val_preds.keys()))
    print("Xem huong dan o cell markdown tren de them v8.")
else:
    m = rmsle(val_trues_vnd, val_preds["v8"])
    print(f"v8 loaded: RMSLE={m:.4f}")


## 3. Alignment check

Dam bao tat ca model pred co cung N va dung voi ground truth.

In [ ]:
N_VAL = len(val_trues_vnd)
print(f"Ground truth val N = {N_VAL}")

for name, arr in list(val_preds.items()):
    if len(arr) != N_VAL:
        print(f"ALIGNMENT ERROR: {name} has {len(arr)} preds, expected {N_VAL}. Removing.")
        del val_preds[name]
    else:
        print(f"  {name:<12}: {len(arr):,} preds OK")

# Cross-verify ground truth consistency
for name, path in PRED_FILES.items():
    if name not in val_preds:
        continue
    trues_from_file = load_trues_from_file(path)
    if trues_from_file is not None and len(trues_from_file) == N_VAL:
        max_diff = float(np.max(np.abs(trues_from_file - val_trues_vnd)))
        status = "OK" if max_diff == 0 else f"DIFF={max_diff:.0f}"
        print(f"  {name:<12}: ground truth check {status}")

print(f"\nModels aligned: {list(val_preds.keys())}")


## 4. Ridge ensemble trong log-space (val)

Fit `Ridge(alpha=0.1)` tren feature matrix `X = [log(pred_i+1)]` → target `y = log(true+1)`.

Ensemble prediction: `log(y_hat+1) = X @ w + b` → `y_hat = exp(X @ w + b) - 1`.

Dung `alpha=0.1` (Tikhonov regularization nhe) — tranh overfit tren 3926 val items.
Also try `alpha` grid de chon tot hon.

In [ ]:
# Build feature matrix X_val
model_names = list(val_preds.keys())
X_val = np.column_stack([
    np.log1p(np.clip(val_preds[name], 0, None))
    for name in model_names
])
y_val_log = np.log1p(val_trues_vnd)

print(f"Feature matrix X_val: {X_val.shape} ({len(model_names)} models x {N_VAL} samples)")
print(f"Models (columns)     : {model_names}")

# Grid search alpha
from sklearn.model_selection import cross_val_score
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]
print("\nAlpha grid search (5-fold CV RMSLE on val):")
for alpha in alphas:
    ridge = Ridge(alpha=alpha, fit_intercept=True)
    scores = cross_val_score(ridge, X_val, y_val_log, cv=5, scoring="neg_root_mean_squared_error")
    print(f"  alpha={alpha:<6}: CV RMSLE log-space = {-scores.mean():.4f} +/- {scores.std():.4f}")

# Fit final Ridge on full val
RIDGE_ALPHA = 0.1
ridge_final = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
ridge_final.fit(X_val, y_val_log)

print(f"\nRidge final (alpha={RIDGE_ALPHA}):")
for name, coef in zip(model_names, ridge_final.coef_):
    print(f"  {name:<12}: coef={coef:.4f}")
print(f"  intercept   : {ridge_final.intercept_:.4f}")


## 5. Eval val — individual + ensemble RMSLE

In [ ]:
# Val RMSLE per model
print("Val RMSLE — individual models:")
for name in model_names:
    m = rmsle(val_trues_vnd, val_preds[name])
    m_all = compute_metrics(val_trues_vnd, val_preds[name])
    print(f"  {name:<12}: RMSLE={m:.4f}  MAE={m_all['mae']:,.0f}  MAPE={m_all['mape']:.1f}%  R2={m_all['r2']:.4f}")

# Ensemble prediction on val
y_ens_val_log = ridge_final.predict(X_val)
y_ens_val = np.expm1(y_ens_val_log)
y_ens_val = np.clip(y_ens_val, 1000, 1_000_000)

ens_metrics_val = compute_metrics(val_trues_vnd, y_ens_val)

print("\nVal RMSLE — ensemble:")
print(f"  ensemble    : RMSLE={ens_metrics_val['rmsle']:.4f}  "
      f"MAE={ens_metrics_val['mae']:,.0f}  "
      f"MAPE={ens_metrics_val['mape']:.1f}%  "
      f"R2={ens_metrics_val['r2']:.4f}")
print("\nBaseline:")
print(f"  v3 (Day5)   : RMSLE=0.4426")
print(f"  Day4 v8     : RMSLE=0.4004")
print(f"  Target P0   : RMSLE<0.40")
print(f"  Target P1   : RMSLE<0.38")


## 6. Test set inference — Qwen models

Load test predictions (neu ton tai) hoac chay inference cho tung Qwen model tren test split.

In [ ]:
test_preds = {}

for name, path in TEST_PRED_FILES.items():
    if name not in val_preds:
        continue  # skip models khong co trong ensemble
    arr = load_preds_from_file(path)
    if arr is not None and len(arr) == len(test_trues_vnd):
        test_preds[name] = arr
        m = rmsle(test_trues_vnd, arr)
        print(f"Loaded test {name:<12}: {len(arr):,} preds, RMSLE={m:.4f}")
    else:
        print(f"MISSING test {name:<12}: {path}")

print(f"\nTest models loaded: {list(test_preds.keys())}")
missing_test = [n for n in model_names if n not in test_preds]
if missing_test:
    print(f"Need to run inference for: {missing_test}")


In [ ]:
# --- [SKIP neu da co tat ca test pred files] ---
missing_test_qwen = [k for k in missing_test if k in QWEN_MODELS]
if not missing_test_qwen:
    print("All test Qwen predictions available. Skip this cell.")
else:
    print(f"Running Qwen test inference for: {missing_test_qwen}")
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    assert torch.cuda.is_available(), "GPU required."
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )

    for name in missing_test_qwen:
        hf_repo = QWEN_MODELS[name]
        print(f"\nInference {name} on test ({len(test_ds)} items)...")
        tok = AutoTokenizer.from_pretrained(hf_repo, trust_remote_code=True)
        tok.pad_token = tok.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3.5-4B-Base",
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            dtype=torch.bfloat16,
        )
        for _m in base.modules():
            if isinstance(_m, torch.nn.Conv1d):
                _m.to(torch.bfloat16)
        mdl = PeftModel.from_pretrained(base, hf_repo, is_trainable=False)
        mdl.eval()

        preds_vnd, dump = [], []
        for i, item in enumerate(tqdm(test_ds, desc=hf_repo[-20:])):
            inputs = tok(item["prompt"], return_tensors="pt").to("cuda")
            with torch.inference_mode():
                out = mdl.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False, pad_token_id=tok.eos_token_id,
                )
            gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            m = re.search(PARSE_REGEX, gen)
            pk = max(PRED_CLAMP_MIN, min(int(float(m.group())), PRED_CLAMP_MAX)) if m else 0
            preds_vnd.append(pk * 1000)
            dump.append({"idx": i, "pred_vnd": pk * 1000, "true_vnd": int(item["price_vnd_true"])})

        save_path = TEST_PRED_FILES[name]
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(dump, f, ensure_ascii=False)
        arr = np.array(preds_vnd, dtype=float)
        test_preds[name] = arr
        m_rmsle = rmsle(test_trues_vnd, arr)
        print(f"  {name}: RMSLE={m_rmsle:.4f} | Saved: {save_path}")

        del mdl, base
        gc.collect()
        torch.cuda.empty_cache()

print(f"\nTest models ready: {list(test_preds.keys())}")


## 7. Ensemble test set + final RMSLE

In [ ]:
# Only use models that have BOTH val + test predictions
ensemble_models = [n for n in model_names if n in test_preds]
if len(ensemble_models) < len(model_names):
    missing_in_test = [n for n in model_names if n not in test_preds]
    print(f"WARNING: Test preds missing for {missing_in_test}. Refit Ridge on {ensemble_models} only.")
    idx = [model_names.index(n) for n in ensemble_models]
    X_val_sub = X_val[:, idx]
    ridge_test = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    ridge_test.fit(X_val_sub, y_val_log)
else:
    ridge_test = ridge_final
    X_val_sub = X_val

N_TEST = len(test_trues_vnd)
X_test = np.column_stack([
    np.log1p(np.clip(test_preds[name], 0, None))
    for name in ensemble_models
])

y_ens_test_log = ridge_test.predict(X_test)
y_ens_test = np.clip(np.expm1(y_ens_test_log), 1000, 1_000_000)

ens_metrics_test = compute_metrics(test_trues_vnd, y_ens_test)

print("=" * 55)
print(f"ENSEMBLE TEST ({N_TEST} items, models={ensemble_models})")
print("=" * 55)
print(f"RMSLE  : {ens_metrics_test['rmsle']:.4f}  (primary)")
print(f"MAE    : {ens_metrics_test['mae']:,.0f} VND")
print(f"MAPE   : {ens_metrics_test['mape']:.1f}%")
print(f"R2     : {ens_metrics_test['r2']:.4f}")
print("=" * 55)

print("\nTest RMSLE per model:")
for name in ensemble_models:
    m = rmsle(test_trues_vnd, test_preds[name])
    print(f"  {name:<12}: RMSLE={m:.4f}")

print("\nBaseline:")
print(f"  Day4 v8 stacked : RMSLE=0.4004")
print(f"  Target P0       : RMSLE<0.40")
print(f"  Target P1       : RMSLE<0.38")


## 8. Plot + Save

In [ ]:
# Plot ensemble val (200 sample)
import numpy as np
rng = np.random.default_rng(SEED)
plot_idx = rng.choice(N_VAL, size=min(200, N_VAL), replace=False)
plot_predictions(
    val_trues_vnd[plot_idx], y_ens_val[plot_idx],
    title=f"Ensemble val ({'+'.join(ensemble_models)})",
)


In [ ]:
results = {
    "ensemble_models": ensemble_models,
    "ridge_alpha": RIDGE_ALPHA,
    "ridge_coefs": {name: float(c) for name, c in zip(ensemble_models, ridge_test.coef_)},
    "ridge_intercept": float(ridge_test.intercept_),
    "val_metrics_per_model": {
        name: compute_metrics(val_trues_vnd, val_preds[name])
        for name in ensemble_models
    },
    "val_metrics_ensemble": ens_metrics_val,
    "test_metrics_per_model": {
        name: compute_metrics(test_trues_vnd, test_preds[name])
        for name in ensemble_models if name in test_preds
    },
    "test_metrics_ensemble": ens_metrics_test,
    "reference": {
        "v3_val_rmsle": 0.4426,
        "v8_test_rmsle": 0.4004,
        "target_p0": 0.40,
        "target_p1": 0.38,
    }
}

with open(ENSEMBLE_RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved: {ENSEMBLE_RESULTS_FILE}")

# Final leaderboard
print("\n" + "=" * 50)
print("FINAL LEADERBOARD — Day 5")
print("=" * 50)
print(f"{'Model':<20} {'Val RMSLE':>10} {'Test RMSLE':>11}")
print("-" * 50)
for name in ensemble_models:
    vm = rmsle(val_trues_vnd, val_preds[name])
    tm = rmsle(test_trues_vnd, test_preds[name]) if name in test_preds else -1
    tm_str = f"{tm:.4f}" if tm >= 0 else "N/A"
    print(f"{name:<20} {vm:>10.4f} {tm_str:>11}")
print("-" * 50)
print(f"{'Ensemble':<20} {ens_metrics_val['rmsle']:>10.4f} {ens_metrics_test['rmsle']:>11.4f}")
print(f"{'Day4 v8 (ref)':<20} {'':>10} {'0.4004':>11}")
print("=" * 50)
